## Regions of interest

Two goals: (1) link the numerical scores back to the actual text, and (2) surface not just the most speech-like *speeches* (expected) but also unusually speech-like *narration* — passages that read more like direct speech than the surrounding narrative without actually being marked as speech (e.g. apostrophe, free indirect discourse).

This picks up where `4 - Compare speechiness methods` leaves off: that notebook establishes that the PCA and dialogism scores agree with each other; this one uses the scores (recomputed here the same way) to explore and present specific passages, rather than to validate the methods themselves.

### Setup

Recomputes the same rolling scores and narrative labels as `4 - Compare speechiness methods`, so this notebook can run independently.

In [1]:
import pandas as pd
from IPython.display import HTML, display

import ccc2026
from ccc2026 import dialogism
ccc2026.setup()

tokens = ccc2026.tokens

In [2]:
lexicons = dialogism.build_lexicons()
dialogism_score = dialogism.token_dialogism_score(lexicons)

feature_set = {
    "lemma": ccc2026.top_lemmas,
    "pos": ccc2026.all_pos,
    "morph": ccc2026.top_morph,
}
train = ccc2026.run_training(feature_set)

ROI_WINDOW = 50
dialogism_roll = dialogism.rolling_dialogism(dialogism_score, window_size=ROI_WINDOW)["speech_score"]["score"]
pca_roll = ccc2026.rolling_samples(train, window_size=ROI_WINDOW)["speech_score"]["score"]

tokens["dialogism_roll"] = dialogism_roll
tokens["pca_roll"] = pca_roll

tokens["label"] = "speech"
tokens.loc[tokens["speaker"] == "Odysseus-Apologue", "label"] = "other"
tokens.loc[tokens["speaker"].isna(), "label"] = "narration"

### Top-scoring regions

In [3]:
def top_regions(label, score_col="dialogism_roll", n=10):
    '''Top n token-level rolling-window positions for a given narrative label, ranked by score.'''
    mask = tokens["label"] == label
    return tokens.loc[mask, score_col].sort_values(ascending=False).head(n)

display_cols = ["work", "line_id", "speaker", "addressee", "text", "pca_roll", "dialogism_roll"]

print("Most speech-like speech:")
display(tokens.loc[top_regions("speech").index, display_cols])

print("Most speech-like narration:")
display(tokens.loc[top_regions("narration").index, display_cols])

Most speech-like speech:


,work,line_id,speaker,addressee,text,pca_roll,dialogism_roll
53014,Iliad,04_0056,Hera,Zeus,ἦ,26.849143,0.547209
53012,Iliad,04_0056,Hera,Zeus,φθονέουσʼ,26.206025,0.544438
53013,Iliad,04_0056,Hera,Zeus,ἐπεὶ,27.028650,0.542818
53015,Iliad,04_0056,Hera,Zeus,πολὺ,26.184692,0.539724
53019,Iliad,04_0057,Hera,Zeus,χρὴ,25.676737,0.539716
53016,Iliad,04_0056,Hera,Zeus,φέρτερός,24.926566,0.538450
53017,Iliad,04_0056,Hera,Zeus,ἐσσι,25.600339,0.537954
53018,Iliad,04_0057,Hera,Zeus,ἀλλὰ,25.431523,0.537634
53011,Iliad,04_0056,Hera,Zeus,ἀνύω,25.558322,0.536950
43005,Iliad,01_0564,Zeus,Hera,ἐμοὶ,31.314852,0.531840


Most speech-like narration:


,work,line_id,speaker,addressee,text,pca_roll,dialogism_roll
208721,Odyssey,16_0308,NaN,NaN,ἀπαμειβόμενος,21.583042,0.507304
199643,Odyssey,14_0148,NaN,NaN,προσέειπε,15.753863,0.506967
208722,Odyssey,16_0308,NaN,NaN,προσεφώνεε,20.780244,0.505538
208723,Odyssey,16_0308,NaN,NaN,φαίδιμος,21.346075,0.504805
208724,Odyssey,16_0308,NaN,NaN,υἱός,21.413463,0.504300
208720,Odyssey,16_0308,NaN,NaN,δʼ,21.057554,0.502643
199642,Odyssey,14_0148,NaN,NaN,αὖτε,14.486121,0.502490
393056,Posthomerica,05_0559,NaN,NaN,προσέειπε,13.133481,0.500242
199641,Odyssey,14_0148,NaN,NaN,δʼ,13.651202,0.499972
196440,Odyssey,13_0146,NaN,NaN,τὸν,14.747488,0.499832


### Highlight speech-like features in the text

In [4]:
def build_display_column(lemma_cutoff=0.5, grammar_cutoff=0.7, lemma_color="red", grammar_color="green"):
    '''Wrap each token's text in a colored <span> if its lemma or any of its
    grammatical features (POS + morph) exceeds the given lexicon cutoff.
    Lemma matches take priority over grammar matches for a given token.
    '''
    lemma_lex = set(lexicons["lemma"].loc[lexicons["lemma"] > lemma_cutoff].index)
    grammar_lex = set(lexicons["grammar"].loc[lexicons["grammar"] > grammar_cutoff].index)

    morph_cols = ["pos", "verbform", "mood", "tense", "voice", "person", "number", "case", "gender"]
    is_lemma_hit = tokens["lemma"].isin(lemma_lex)
    is_grammar_hit = tokens[morph_cols].isin(grammar_lex).any(axis=1)

    text = tokens["text"]
    display_col = text.copy()
    display_col = display_col.where(~is_grammar_hit, f'<span style="color:{grammar_color}">' + text + "</span>")
    display_col = display_col.where(~is_lemma_hit, f'<span style="color:{lemma_color}">' + text + "</span>")
    return display_col

tokens["display"] = build_display_column()

### Inline HTML view, one work at a time

In [5]:
TABLE_COLUMNS = ["pref", "line", "label", "speaker", "text", "pca", "dialogism"]

def _row_to_html_tr(row):
    cells = "".join(f"<td>{row[col]}</td>" for col in TABLE_COLUMNS)
    return f'<tr style="text-align:left">{cells}</tr>'

def region_table_html(work):
    '''Build an HTML table of a work's lines, with highlighted text and mean scores per line.'''
    lines = tokens.groupby(["work", "line_id"], sort=False).agg(
        pref = ("pref", "first"),
        line = ("line", "first"),
        label = ("label", "first"),
        speaker = ("speaker", "first"),
        text = ("display", " ".join),
        pca = ("pca_roll", "mean"),
        dialogism = ("dialogism_roll", "mean"),
    ).loc[work]

    lines["pca"] = lines["pca"].round(2)
    lines["dialogism"] = lines["dialogism"].round(3)

    head = "<thead><tr>" + "".join(f"<th>{col}</th>" for col in TABLE_COLUMNS) + "</tr></thead>"
    rows = "\n".join(_row_to_html_tr(row) for _, row in lines.iterrows())
    # a plain "style" attribute on <table> doesn't cascade into td/th in Jupyter's
    # own notebook CSS, which sets its own (right-aligned) td/th rule — so target
    # td/th directly, scoped to this table's class, rather than adding a style
    # attribute to every single cell
    style = '<style>table.roi-table td, table.roi-table th { text-align: left !important; }</style>'
    return style + f'<table class="roi-table">{head}<tbody>{rows}</tbody></table>'

In [6]:
display(HTML(region_table_html("Iliad")))

pref,line,label,speaker,text,pca,dialogism
1,1,narration,nan,μῆνιν ἄειδε θεὰ Πηληϊάδεω Ἀχιλῆος,nan,nan
1,2,narration,nan,οὐλομένην ἣ μυρίʼ Ἀχαιοῖς ἄλγεʼ ἔθηκε,-5.85,0.341
1,3,narration,nan,πολλὰς δʼ ἰφθίμους ψυχὰς Ἄϊδι προΐαψεν,-8.22,0.341
1,4,narration,nan,ἡρώων αὐτοὺς δὲ ἑλώρια τεῦχε κύνεσσιν,-6.47,0.347
1,5,narration,nan,οἰωνοῖσί τε πᾶσι Διὸς δʼ ἐτελείετο βουλή,-6.26,0.353
1,6,narration,nan,ἐξ οὗ δὴ τὰ πρῶτα διαστήτην ἐρίσαντε,-4.83,0.358
1,7,narration,nan,Ἀτρεΐδης τε ἄναξ ἀνδρῶν καὶ δῖος Ἀχιλλεύς,-4.31,0.358
1,8,narration,nan,τίς τʼ ἄρ σφωε θεῶν ἔριδι ξυνέηκε μάχεσθαι,-4.06,0.363
1,9,narration,nan,Λητοῦς καὶ Διὸς υἱός ὃ γὰρ βασιλῆϊ χολωθεὶς,-4.25,0.366
1,10,narration,nan,νοῦσον ἀνὰ στρατὸν ὄρσε κακήν ὀλέκοντο δὲ λαοί,-3.08,0.369


## Export the whole corpus to Excel

Each work gets its own worksheet, one row per verse line, with per-token
colored rich text (same lemma/grammar highlighting as the inline HTML view
above) and per-line scores. This replaces the clipboard-based copy-paste
workflow — no OS clipboard dependency, works the same in Colab as locally,
and produces a real file you can open, download, or email.

In [7]:
from openpyxl import Workbook
from openpyxl.cell.rich_text import CellRichText, TextBlock
from openpyxl.cell.text import InlineFont
from openpyxl.formatting.rule import ColorScaleRule

HEADERS = ["pref", "line", "label", "speaker", "addressee", "text", "pca", "dialogism"]
SCORE_COLUMNS = {"pca": "G", "dialogism": "H"}

def _line_rich_text(group, lemma_lex, grammar_lex, morph_cols, lemma_color="FF0000", grammar_color="008000"):
    '''Build one CellRichText for a line's tokens, coloring each token that hits
    the lemma or grammar lexicon. Lemma matches take priority over grammar
    matches, same as the inline HTML highlighter.'''
    blocks = []
    rows = list(group.iterrows())
    for i, (_, tok) in enumerate(rows):
        piece = tok["text"] + (" " if i < len(rows) - 1 else "")
        if tok["lemma"] in lemma_lex:
            blocks.append(TextBlock(InlineFont(color=lemma_color), piece))
        elif any(pd.notna(tok[c]) and tok[c] in grammar_lex for c in morph_cols):
            blocks.append(TextBlock(InlineFont(color=grammar_color), piece))
        else:
            blocks.append(piece)
    return CellRichText(*blocks)

def _add_score_color_scale(ws, col_letter):
    '''Blue (low) - white (median) - red (high) color scale, matching Excel's
    built-in preset. A fresh Rule instance per call, since a Rule shouldn't be
    reused across multiple conditional_formatting ranges.'''
    rule = ColorScaleRule(
        start_type="min", start_color="5A8AC6",
        mid_type="percentile", mid_value=50, mid_color="FCFCFF",
        end_type="max", end_color="F8696B",
    )
    ws.conditional_formatting.add(f"{col_letter}2:{col_letter}{ws.max_row}", rule)

def export_regions_to_excel(filename, lemma_cutoff=0.5, grammar_cutoff=0.7):
    '''Export the whole corpus to an .xlsx workbook, one worksheet per work,
    one row per verse line, with rich-text colored tokens, per-line scores,
    and a color scale on the score columns (helps see at a glance how the
    two measures track each other, and where they line up with speech
    boundaries).
    '''
    lemma_lex = set(lexicons["lemma"].loc[lexicons["lemma"] > lemma_cutoff].index)
    grammar_lex = set(lexicons["grammar"].loc[lexicons["grammar"] > grammar_cutoff].index)
    morph_cols = ["pos", "verbform", "mood", "tense", "voice", "person", "number", "case", "gender"]

    wb = Workbook()
    wb.remove(wb.active)  # drop the default empty sheet

    for work_name, _ in ccc2026.CONFIG["texts"]:
        ws = wb.create_sheet(title=work_name[:31])  # Excel sheet-name length limit
        ws.append(HEADERS)

        work_tokens = tokens.loc[tokens["work"] == work_name]
        for line_id, group in work_tokens.groupby("line_id", sort=False):
            row0 = group.iloc[0]
            rich_text = _line_rich_text(group, lemma_lex, grammar_lex, morph_cols)
            ws.append([
                row0["pref"], row0["line"], row0["label"], row0["speaker"], row0["addressee"],
                rich_text, group["pca_roll"].mean(), group["dialogism_roll"].mean(),
            ])

        for col_letter in SCORE_COLUMNS.values():
            _add_score_color_scale(ws, col_letter)

    wb.save(filename)

export_regions_to_excel("regions_of_interest.xlsx")